In [182]:
## IMPORTS
import random
import imageio
import numpy as np
import yaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import einops
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from torchvision.transforms import Compose, ToTensor, Lambda, Normalize
from torchvision.datasets.mnist import MNIST
from torchvision.datasets import CIFAR10

from math import exp, sqrt

device="cuda" if torch.cuda.is_available() else "cpu"

##Paramètres de reproductibilité

In [183]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


##Paramètres de diffusion

In [ ]:
n_steps = 1000
t_linspace, dt = range(n_steps+1),1
time_emb_dim = 100
sigma = 0.1
beta = 2
lambda_t = torch.tensor([1-i/n_steps for i in range(n_steps)]).to(device)

## Réseaux Unet

In [185]:
def sinusoidal_embedding(n, d):
    # Returns the standard positional embedding
    embedding = torch.zeros(n, d)
    wk = torch.tensor([1 / 10_000 ** (2 * j / d) for j in range(d)])
    wk = wk.reshape((1, d))
    t = torch.arange(n).reshape((n, 1))
    embedding[:, ::2] = torch.sin(t * wk[:, ::2])
    embedding[:, 1::2] = torch.cos(t * wk[:, ::2])

    return embedding


class MyBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1, activation=None, normalize=True):
        super(MyBlock, self).__init__()
        self.norm = nn.GroupNorm(num_groups=self._get_num_groups(in_c), num_channels=in_c)
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size, stride, padding)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size, stride, padding)
        self.activation = nn.SiLU() if activation is None else activation
        self.normalize = normalize

    def forward(self, x):
        out = self.norm(x) if self.normalize else x
        out = self.conv1(out)
        out = self.activation(out)
        out = self.conv2(out)
        out = self.activation(out)
        return out

    def _get_num_groups(self, num_channels):
        """ Retourne le plus grand nombre de groupes possible tout en restant <= 8 et divisible par num_channels """
        for g in range(min(num_channels, 8), 0, -1):
            if num_channels % g == 0:
                return g
        return 1  # Fallback pour éviter les erreurs

###Unet pour Mnist

In [186]:
class MyUNet_Mnist(nn.Module):
    # adapté à MNIST seulement
    def __init__(self, im_size=28, channels=1, n_steps=n_steps,
                 time_emb_dim=time_emb_dim):
        super().__init__()

        self.im_size = im_size  # Taille de l'image (hauteur et largeur)
        self.channels = channels  # Nombre de canaux d'entrée (par exemple 1 pour MNIST)

        # Sinusoidal embedding
        self.time_embed = nn.Embedding(n_steps, time_emb_dim)
        self.time_embed.weight.data = sinusoidal_embedding(n_steps, time_emb_dim)
        self.time_embed.requires_grad_(False)

        # Calcul de la taille de l'image à chaque downsampling
        self.height, self.width = im_size, im_size
        self.downsampling_factor = [2, 2, 2, 2]  # Facteur de downsampling

        # Premier half
        self.te1 = self._make_te(time_emb_dim, 1)
        self.b1 = nn.Sequential(
            MyBlock(channels, 10),
            MyBlock(10, 10),
            MyBlock(10, 10)
        )
        self.down1 = nn.Conv2d(10, 10, 4, 2, 1)

        self.te2 = self._make_te(time_emb_dim, 10)
        self.b2 = nn.Sequential(
            MyBlock(10, 20),
            MyBlock(20, 20),
            MyBlock(20, 20)
        )
        self.down2 = nn.Conv2d(20, 20, 4, 2, 1)

        self.te3 = self._make_te(time_emb_dim, 20)
        self.b3 = nn.Sequential(
            MyBlock(20, 40),
            MyBlock(40, 40),
            MyBlock(40, 40)
        )
        self.down3 = nn.Sequential(
            nn.Conv2d(40, 40, 2, 1),
            nn.SiLU(),
            nn.Conv2d(40, 40, 4, 2, 1)
        )

        # Nouveau bloc Down
        self.te4 = self._make_te(time_emb_dim, 40)
        self.b4 = nn.Sequential(
            MyBlock(40, 80),
            MyBlock(80, 80),
            MyBlock(80, 80)
        )
        self.down4 = nn.Sequential(
            nn.Conv2d(80, 80, 2, 1),
            nn.SiLU(),
            nn.Conv2d(80, 80, 4, 2, 1)
        )

        # Bottleneck
        self.te_mid = self._make_te(time_emb_dim, 80)
        self.b_mid = nn.Sequential(
            MyBlock(80, 40),
            MyBlock(40, 40),
            MyBlock(40, 80)
        )

        # Second half (Upsampling)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(80, 80, 4, 2, 1),
            nn.SiLU(),
            nn.ConvTranspose2d(80, 80, 2, 1)
        )

        self.te5 = self._make_te(time_emb_dim, 160)
        self.b5 = nn.Sequential(
            MyBlock(160, 80),
            MyBlock(80, 40),
            MyBlock(40, 40)
        )

        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(40, 40, 4, 2, 1),
            nn.SiLU(),
            nn.ConvTranspose2d(40, 40, 2, 1)
        )

        self.te6 = self._make_te(time_emb_dim, 80)
        self.b6 = nn.Sequential(
            MyBlock(80, 40),
            MyBlock(40, 20),
            MyBlock(20, 20)
        )

        self.up2 = nn.ConvTranspose2d(20, 20, 4, 2, 1)
        self.te7 = self._make_te(time_emb_dim, 40)
        self.b7 = nn.Sequential(
            MyBlock(40, 20),
            MyBlock(20, 10),
            MyBlock(10, 10)
        )

        self.up3 = nn.ConvTranspose2d(10, 10, 4, 2, 1)
        self.te_out = self._make_te(time_emb_dim, 20)
        self.b_out = nn.Sequential(
            MyBlock(20, 10),
            MyBlock(10, 10),
            MyBlock(10, 10, normalize=False)
        )

        self.conv_out = nn.Conv2d(10, channels, 3, 1, 1)

    def forward(self, x, t):
        # x is (N, C, H, W) where C is channels, H and W are height and width
        t = self.time_embed(t)
        n = len(x)
        out1 = self.b1(x + self.te1(t).reshape(n, -1, 1, 1))
        out2 = self.b2(self.down1(out1) + self.te2(t).reshape(n, -1, 1, 1))
        out3 = self.b3(self.down2(out2) + self.te3(t).reshape(n, -1, 1, 1))
        out4 = self.b4(self.down3(out3) + self.te4(t).reshape(n, -1, 1, 1))

        out_mid = self.b_mid(self.down4(out4) + self.te_mid(t).reshape(n, -1, 1, 1))

        out5 = torch.cat((out4, self.up0(out_mid)), dim=1)
        out5 = self.b5(out5 + self.te5(t).reshape(n, -1, 1, 1))

        out6 = torch.cat((out3, self.up1(out5)), dim=1)
        out6 = self.b6(out6 + self.te6(t).reshape(n, -1, 1, 1))

        out7 = torch.cat((out2, self.up2(out6)), dim=1)
        out7 = self.b7(out7 + self.te7(t).reshape(n, -1, 1, 1))

        out = torch.cat((out1, self.up3(out7)), dim=1)
        out = self.b_out(out + self.te_out(t).reshape(n, -1, 1, 1))

        out = self.conv_out(out)

        return out

    def _make_te(self, dim_in, dim_out):
        return nn.Sequential(
            nn.Linear(dim_in, dim_out),
            nn.SiLU(),
            nn.Linear(dim_out, dim_out)
        )

###Unet pour CIFAR

In [187]:
class MyUNet_cifar(nn.Module):
    # adapté à CIFAR
    def __init__(self, im_size=32, channels=3, n_steps=n_steps,
                 time_emb_dim=time_emb_dim):
        super().__init__()

        self.im_size = im_size
        self.channels = channels

        self.time_embed = nn.Embedding(n_steps, time_emb_dim)
        self.time_embed.weight.data = sinusoidal_embedding(n_steps, time_emb_dim)
        self.time_embed.requires_grad_(False)
        # Downsampling
        self.te1 = self._make_te(time_emb_dim, 3)
        self.b1 = MyBlock(channels, 32)
        self.down1 = nn.Conv2d(32, 32, 4, 2, 1)  # 32x32 -> 16x16

        self.te2 = self._make_te(time_emb_dim, 32)
        self.b2 = MyBlock(32, 64)
        self.down2 = nn.Conv2d(64, 64, 4, 2, 1)  # 16x16 -> 8x8

        self.te3 = self._make_te(time_emb_dim, 64)
        self.b3 = MyBlock(64, 128)
        self.down3 = nn.Conv2d(128, 128, 4, 2, 1)  # 8x8 -> 4x4

        self.te4 = self._make_te(time_emb_dim, 128)
        self.b4 = MyBlock(128, 256)

        # Bottleneck
        self.te_mid = self._make_te(time_emb_dim, 256)
        self.b_mid = MyBlock(256, 128)

        # Upsampling
        self.up1 = nn.ConvTranspose2d(128, 128, 4, 2, 1)  # 4x4 -> 8x8
        self.te5 = self._make_te(time_emb_dim, 256)
        self.b5 = MyBlock(256, 64)

        self.up2 = nn.ConvTranspose2d(64, 64, 4, 2, 1)  # 8x8 -> 16x16
        self.te6 = self._make_te(time_emb_dim, 128)
        self.b6 = MyBlock(128, 32)

        self.up3 = nn.ConvTranspose2d(32, 32, 4, 2, 1)  # 16x16 -> 32x32
        self.te7 = self._make_te(time_emb_dim, 64)
        self.b7 = MyBlock(64, 32)

        self.conv_out = nn.Conv2d(32, channels, 3, 1, 1)

    def forward(self, x, t):
        t = self.time_embed(t)
        n = len(x)

        out1 = self.b1(x + self.te1(t).reshape(n, -1, 1, 1))
        out2 = self.b2(self.down1(out1) + self.te2(t).reshape(n, -1, 1, 1))
        out3 = self.b3(self.down2(out2) + self.te3(t).reshape(n, -1, 1, 1))
        out4 = self.b4(self.down3(out3) + self.te4(t).reshape(n, -1, 1, 1))

        out_mid = self.b_mid(out4 + self.te_mid(t).reshape(n, -1, 1, 1))
        print(out_mid.shape)
        print(out4.shape)
        out5 = torch.cat((out4, self.up1(out_mid)), dim=1)
        out5 = self.b5(out5 + self.te5(t).reshape(n, -1, 1, 1))

        out6 = torch.cat((out3, self.up2(out5)), dim=1)
        out6 = self.b6(out6 + self.te6(t).reshape(n, -1, 1, 1))

        out7 = torch.cat((out2, self.up3(out6)), dim=1)
        out7 = self.b7(out7 + self.te7(t).reshape(n, -1, 1, 1))

        out = self.conv_out(out7)
        return out

    def _make_te(self, dim_in, dim_out):
        return nn.Sequential(
            nn.Linear(dim_in, dim_out),
            nn.SiLU(),
            nn.Linear(dim_out, dim_out)
        )

##Modèle SSM

In [ ]:
class MySSM(nn.Module):

    def __init__(self):

        super().__init__()
        self.device = device
        if db == "Cifar" :
          self.network = MyUNet_cifar().to(device)
          self.image_chw = (3, 32, 32)
        else :
          self.network = MyUNet_Mnist().to(device)
          self.image_chw = (1, 28, 28) #Par défaut il fait du Mnist

        self.gamma_t = np.array([exp(-(beta*t)) for t in t_linspace])
        sigma_t_squared = np.array([(sigma**2 / 2*beta) * (1 - exp(-(2*beta*t))) for t in t_linspace])
        self.sigma_t = np.array([sqrt(s) for s in sigma_t_squared])

    def forward(self, x0, t, eta=None):
        # transforme l'input (x0) en image bruitée au temps t (passé en argument), avec le bruit eta
        n, c, h, w = x0.shape

        if eta is None:
            eta = torch.randn(n, c, h, w).to(device)

        i=random.randint(0,n_steps)
        noisy = self.gamma_t[i]*x0 + self.sigma_t[i]*eta
        return noisy

    def backward(self, x, t):
        return self.network(x, t)  # envoie x, t dans le UNET. En sortie, le bruit estimé sur x au temps t

    def generate(self, n_samples=4, frames_per_gif=100, gif_name="sampling.gif") :
      """Given a SSM model, a number of samples to be generated and a device, returns some newly generated samples"""
      frame_idxs = np.linspace(0, n_steps, frames_per_gif).astype(np.uint)
      frames = []
      c, h, w = self.image_chw

      with torch.no_grad():

          # Starting from random noise
          x = torch.randn(n_samples, c, h, w).to(device)

          for idx, t in enumerate(list(range(n_steps))[::-1]):
              # Estimating noise to be removed
              time_tensor = (torch.ones(n_samples, ) * t).to(device).long()
              s_theta = self.backward(x, time_tensor).to(device)

              # Denoising the image
              x = x + dt * (s_theta * sigma ** 2 + beta * x)  + sigma * sqrt(dt) * torch.randn_like(x).to(device)
              print(x)

              # Adding frames to the GIF
              if idx in frame_idxs :
                  # Putting digits in range [0, 255]
                  normalized = x.clone()
                  for i in range(len(normalized)):
                      normalized[i] -= torch.min(normalized[i])
                      normalized[i] *= 255 / torch.max(normalized[i])

                  # Reshaping batch (n, c, h, w) to be a (as much as it gets) square frame
                  frame = einops.rearrange(normalized, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=int(n_samples ** 0.5))
                  frame = frame.cpu().numpy().astype(np.uint8)

                  # Rendering frame
                  frames.append(frame)

      # Storing the gif
      with imageio.get_writer(gif_name, mode="I") as writer:
          for idx, frame in enumerate(frames):
              rgb_frame = np.repeat(frame, 3, axis=2)
              writer.append_data(rgb_frame)

              # Showing the last frame for a longer time
              if idx == len(frames) - 1:
                  last_rgb_frame = np.repeat(frames[-1], 3, axis=2)
                  for _ in range(frames_per_gif // 3):
                      writer.append_data(last_rgb_frame)

      return x



##Affichage d'images

In [189]:
def show_images(images, title=""):
    """Shows the provided images as sub-pictures in a square"""

    import matplotlib.pyplot as plt
    import numpy as np

    # Converting images to CPU numpy arrays
    if isinstance(images, torch.Tensor):
        images = images.detach().cpu().numpy()

    # Détection du nombre de canaux
    channels = images.shape[1]  # (batch_size, channels, height, width)

    # Si c'est une image en niveau de gris, on enlève la dimension des canaux
    if channels == 1:
        images = images[:, 0, :, :]  # (batch_size, height, width)
    else:
        # Transposer (C, H, W) → (H, W, C) pour plt.imshow()
        images = np.transpose(images, (0, 2, 3, 1))

    # Defining number of rows and columns
    fig = plt.figure(figsize=(8, 8))
    rows = int(len(images) ** (1 / 2))
    cols = round(len(images) / rows)

    # Remet les pixels dans [0, 1] en inversant la normalisation
    images = (images + 1) / 2

    # Populating figure with sub-plots
    idx = 0
    for r in range(rows):
        for c in range(cols):
            fig.add_subplot(rows, cols, idx + 1)

            if idx < len(images):
                plt.imshow(images[idx], cmap="gray" if channels == 1 else None)
                plt.axis('off')
                idx += 1

    fig.suptitle(title, fontsize=30)

    # Showing the figure
    plt.show()

##Entrainement du modèle

In [ ]:
def training_loop(model, optim, display=False):
    mse = nn.MSELoss()
    best_loss = float("inf")

    if db == "Cifar":
        transform = Compose([
        ToTensor(),  # Convertit en tensor et normalise les pixels dans [0, 1]
        Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalisation RGB pour CIFAR-10
    ])
        dataset = CIFAR10(root='./datasets', download=True, train=True, transform=transform)
    else :
        transform = Compose([
        ToTensor(),
        Lambda(lambda x: (x - 0.5) * 2)]
    )
        dataset = MNIST("./datasets", download=True, train=True, transform=transform)

    dataset, _ = torch.utils.data.random_split(dataset,[500, len(dataset)-500]) #Juste pour prendre moins de données pour le débuggage

    loader = DataLoader(dataset, batch_size, shuffle=True)

    f=True

    for epoch in tqdm(range(n_epochs), desc=f"Training progress", colour="#00ff00"):
        epoch_loss = 0.0
        for step, batch in enumerate(tqdm(loader, leave=False, position=0,
                                          desc=f"Epoch {epoch + 1}/{n_epochs}", colour="#005500")):
            # Loading data
            x0 = batch[0].to(device)
            n = len(x0) # taille effective du batch
            k = torch.randint(0, n_steps, (n,)).to(device) # step aléatoire pour chaque image x0 du batch
            t = dt*k

            # appel à forward pour obtenir le batch bruité
            xt = model(x0, t)

            #Calcul de l'estimateur de Hutchinson (avec.grad, ne marche pas)
            # nu = torch.randn_like(x0).to(device)
            # h = 0. * torch.ones((n,1,1,1)).to(device).detach().requires_grad_(True)
            # Hutchinson = torch.sum(model.backward(xt + nu*h, t.reshape(n, -1)) * nu, dim=(1, 2, 3)) #nu^T s_theta(xt+nu*h,t)
            # D_Hutchinson = torch.autograd.grad(Hutchinson,h,torch.ones_like(Hutchinson),create_graph=True)[0]

            ###Avec calcul de Jacobienne directement (très long)
            # nu = torch.randn_like(x0).to(device)
            # #h = 0. * torch.ones(n).to(device).detach().requires_grad_(True)
            # h = torch.tensor(0.).to(device).detach().requires_grad_(True)
            # #new_h=h.reshape((n,1,1,1))

            # new_nu=nu.flatten(1,-1)
            # #if f : print(new_nu.shape) ; f=False
            # def Hutchinson(h) : return torch.sum(model.backward(xt + nu*h, t.reshape(n, -1)).flatten(1,-1) * new_nu,dim=1)
            # #Hutchinson = torch.sum(model.backward(xt + nu*h, t.reshape(n, -1)).flatten(1,-1) * new_nu,dim=1)#.reshape((n,1,1,1)) #nu^T s_theta(xt+nu*h,t)
            # #if f : print(Hutchinson.shape) ; f=False
            # D_Hutchinson = torch.autograd.functional.jacobian(Hutchinson,h,create_graph=True)
            # if f : print(D_Hutchinson.shape) ; f=False



            ###Avec la fonction Jacobienne (très long)
            # nu = torch.randn_like(x0).to(device)
            # #h = 0. * torch.ones(n).to(device).detach().requires_grad_(True)
            # new_nu=nu.flatten(1,-1)

            # def Hutchinson(h) : return torch.sum(model.backward(xt + nu*h, t.reshape(n, -1)).flatten(1,-1) * new_nu,dim=1)
            # D_Hutchinson_fonc=torch.func.jacrev(Hutchinson)
            # h = torch.tensor(0.).to(device).detach().requires_grad_(True)
            # D_Hutchinson = D_Hutchinson_fonc(h)
            # if f : print(D_Hutchinson.shape) ; f=False
            #new_h=h.reshape((n,1,1,1))


            #if f : print(new_nu.shape) ; f=False

            #Hutchinson = torch.sum(model.backward(xt + nu*h, t.reshape(n, -1)).flatten(1,-1) * new_nu,dim=1)#.reshape((n,1,1,1)) #nu^T s_theta(xt+nu*h,t)
            #if f : print(Hutchinson.shape) ; f=False


            ###Avec une boucle sur le batch (très long)
            nu = torch.randn_like(x0).to(device)
            D_Hutchinson = torch.zeros(n).to(device)
            for i,x_i in enumerate(xt) :
              nu_i = nu[i]
              h = torch.tensor(0.).to(device).detach().requires_grad_(True)
              score = model.backward(x_i + nu_i*h, t[i].reshape(1, -1))
              Hutchinson = nu_i.flatten()@score.flatten()
              Hutchinson.backward()
              D_Hutchinson[i] = h.grad


            ###Sans passer par la simplification en dérivé (Google colab trouve que ça prend trop de mémoire....)
            # nu = torch.randn_like(x0).to(device)
            # def Hutchinson(x_t,t,nu) :
            #   dim_flatten=np.prod(model.image_chw)
            #   jac=torch.func.jacrev(model.backward,0)(x_t,t).reshape((dim_flatten,dim_flatten))
            #   nu=nu.flatten()
            #   return nu@jac@nu
            # D_Hutchinson_fonc=torch.vmap(Hutchinson)
            # D_Hutchinson = D_Hutchinson_fonc(xt,t,nu)
            
            # estimation du bruit par le modèle (backward)
            s_theta = model.backward(xt, t.reshape(n, -1))

            lambda_t = 1-t/n_steps

            # loss : mse entre bruit prédit et bruit réel
            loss = torch.mean(lambda_t*(torch.sum(s_theta**2,dim=(1, 2, 3)) + 2 * D_Hutchinson ))
            optim.zero_grad()
            loss.backward()
            optim.step()

            epoch_loss += loss.item() * len(x0) / len(loader.dataset) # loss moyenne de l'epoch

        # affichage de l'image générée à cet epoch
        if display:
            show_images(model.generate(), f"Généré à l'epoch {epoch + 1}")

        log_string = f"Loss epoch {epoch + 1}: {epoch_loss:.3f}"

        print(log_string)

##Paramètres d'apprentissage

In [ ]:
batch_size = 8
n_epochs = 2
num_grid_rows = 1
lr = 0.001
train_display = False
db="Mnist" #Ou Cifar

##Exécution

In [192]:
ssm = MySSM()
optim = torch.optim.Adam(ssm.network.parameters(), lr=lr)
training_loop(ssm, optim, train_display)

Training progress:   0%|          | 0/2 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 188.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 14.12 MiB is free. Process 5360 has 14.72 GiB memory in use. Of the allocated memory 13.56 GiB is allocated by PyTorch, and 1.02 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)